In [ ]:
import os, random
import numpy as np
import torch
import matplotlib.pyplot as plt
from cf_data import make_loaders
from squid_bipartite import BipartiteSQUIDGNNPowerControl
from train_utils import train_one_epoch, evaluate, compute_eta_baselines, train_one_epoch_semi_fixed

In [ ]:
CFG = dict(
    mat_path="Data/cf_30_6_cvx_wmmse.mat",
    batch_size=16,          # SQUID/PennyLane is slow; start small
    test_batch_size=64,
    epochs=100,
    lr=7e-3,
    top_l_aps=6,
    objective="sum",
    model="squid",         # "squid" or "gnn"
    q_layers=2,
    mp_layers=2,
    seed=7,
    save_dir="runs_cf_squid_bipartite",
)
device = torch.device("cpu")
os.makedirs(CFG["save_dir"], exist_ok=True)
random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])

In [ ]:
train_loader, test_loader, meta = make_loaders(
    train_mat=CFG["mat_path"],
    test_mat=None,
    train_ratio=0.8,
    num_samples=50,
    batch_size=CFG["batch_size"],
    test_batch_size=100,
    top_l_aps=6,
    seed=7,
    use_log_beta=False
)

meta

In [ ]:
from uplink.squid_bipartite_attention import BipartiteSQUIDGNNPowerControl_Attention

model = BipartiteSQUIDGNNPowerControl_Attention(
    ap_in=meta["ap_dim"],
    ue_in=meta["ue_dim"],
    edge_dim=meta["edge_dim"],
    hidden_dim=32,
    pqc_dim=2,
    top_l_aps=4,
    top_l_ues=4,
    n_qubits_amp=None,
    q_layers=1,
    mp_layers=3,
    q_device="default.qubit",
    dropout=0.1,
    residual_scale=0.8,
    noise_std=0.01,
    output_temperature=1.0,
    output_bias_init=-1.0,
    use_softmax_power=False,
    edge_scale=5.0,
    epsilon_random=0.1,
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=CFG["lr"])
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.8)

In [ ]:
cvx_base, wmmse_base = compute_eta_baselines(test_loader, device, CFG["objective"])
cvx_base_train, wmmse_base_train = compute_eta_baselines(train_loader, device, CFG["objective"])
print("CVX eta baseline:", cvx_base)
print("WMMSE eta baseline:", wmmse_base)

In [ ]:
history = []
best_test = -1e9

for epoch in range(1000):
    train_info = train_one_epoch_semi_fixed(
        model=model,
        optimizer=optimizer,
        loader=train_loader,
        device=device,
        objective=CFG["objective"],
        lambda_mse=0.5,
        lambda_rate=0.5,
    )

    test_metric = evaluate(
        model,
        test_loader,
        device,
        CFG["objective"],
    )

    history.append([
        epoch,
        train_info["loss"],
        train_info["mse"],
        train_info["rate_loss"],
        test_metric,
    ])

    print(
        f"Epoch {epoch:03d} | "
        f"loss={train_info['loss']:.4f} | "
        f"mse={train_info['mse']:.4f} | "
        f"rate_loss={train_info['rate_loss']:.4f} | "
        f"test_rate={test_metric:.4f} | "
    )

In [ ]:
@torch.no_grad()
def collect_model_outputs(model, loader, device, max_batches=None):
    model.eval()
    outputs = []

    for b, batch in enumerate(loader):
        data = batch[0].to(device)

        eta = model(
            data.x_dict,
            data.edge_index_dict,
            data.edge_attr_dict,
        )

        outputs.append(eta.detach().cpu().reshape(-1))

        if max_batches is not None and b + 1 >= max_batches:
            break

    return torch.cat(outputs).numpy()


eta_out = collect_model_outputs(model, test_loader, device)

plt.figure(figsize=(7, 4))
plt.hist(eta_out, bins=30, edgecolor="black", alpha=0.75)
plt.xlabel(r"Predicted power coefficient $\eta_k$")
plt.ylabel("Count")
plt.title("Distribution of model outputs")
plt.grid(True)
plt.tight_layout()
plt.show()

print("min:", eta_out.min())
print("max:", eta_out.max())
print("mean:", eta_out.mean())
print("std:", eta_out.std())

In [ ]:
def collect_baseline_eta(loader, name="eta_wmmse"):
    vals = []
    for batch in loader:
        extra = batch[3]
        if name in extra:
            vals.append(extra[name].reshape(-1))
    return torch.cat(vals).numpy()

eta_w = collect_baseline_eta(test_loader, "eta_wmmse")
eta_c = collect_baseline_eta(test_loader, "eta_cvx")

plt.figure(figsize=(7,4))
plt.hist(eta_w, bins=30, alpha=0.6, label="WMMSE")
plt.hist(eta_c, bins=30, alpha=0.6, label="CVX")
plt.xlabel("Power coefficient")
plt.ylabel("Count")
plt.legend()
plt.grid(True)
plt.show()

print("WMMSE mean:", eta_w.mean(), "CVX mean:", eta_c.mean())

In [ ]:
history_np = np.asarray(history)

plt.figure(figsize=(7,4))
epochs = np.arange(len(history_np[:,0]))
plt.plot(epochs, -history_np[:,3], label="Train rate")
plt.plot(epochs, history_np[:,4], label="Test rate")
plt.axhline(cvx_base, linestyle="--", linewidth=2, label=f"CVX test ({cvx_base:.4f})", color="red")
plt.axhline(cvx_base_train, linestyle=":", linewidth=2, label=f"CVX train ({cvx_base_train:.4f})", color="red")
plt.axhline(wmmse_base, linestyle="-.", linewidth=2, label=f"WMMSE test ({wmmse_base:.4f})", color="blue")
plt.axhline(wmmse_base_train, linestyle=":", linewidth=2, label=f"WMMSE train ({wmmse_base_train:.4f})", color="blue")
plt.xlabel("Epoch")
plt.ylabel(f"{CFG['objective']}-rate")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# QAttention

In [ ]:
from uplink.squid_bipartite_Qattention import BipartiteSQUIDGNNPowerControl_QAttention

model_q = BipartiteSQUIDGNNPowerControl_QAttention(
    ap_in=meta["ap_dim"],
    ue_in=meta["ue_dim"],
    edge_dim=meta["edge_dim"],
    hidden_dim=32,
    pqc_dim=2,
    top_l_aps=4,
    top_l_ues=4,
    n_qubits_amp=None,
    q_layers=1,
    mp_layers=3,
    q_device="default.qubit",
    dropout=0.1,
    residual_scale=0.8,
    noise_std=0.01,
    output_temperature=1.0,
    output_bias_init=-1.0,
    use_softmax_power=False,
    edge_scale=5.0,
    epsilon_random=0.1,
).to(device)
optimizer_q = torch.optim.Adam(model_q.parameters(), lr=CFG["lr"])
scheduler_q = torch.optim.lr_scheduler.StepLR(optimizer_q, step_size=50, gamma=0.8)

In [ ]:
cvx_base, wmmse_base = compute_eta_baselines(test_loader, device, CFG["objective"])
cvx_base_train, wmmse_base_train = compute_eta_baselines(train_loader, device, CFG["objective"])

In [ ]:
history_q = []
best_test = -1e9

for epoch in range(1000):
    train_info = train_one_epoch_semi_fixed(
        model=model_q,
        optimizer=optimizer_q,
        loader=train_loader,
        device=device,
        objective=CFG["objective"],
        lambda_mse=0.5,
        lambda_rate=0.5,
    )

    test_metric = evaluate(
        model_q,
        test_loader,
        device,
        CFG["objective"],
    )

    history_q.append([

     epoch,
        train_info["loss"],
        train_info["mse"],
        train_info["rate_loss"],
        test_metric,
    ])

    print(
        f"Epoch {epoch:03d} | "
        f"loss={train_info['loss']:.4f} | "
        f"mse={train_info['mse']:.4f} | "
        f"rate_loss={train_info['rate_loss']:.4f} | "
        f"test_rate={test_metric:.4f} | "
    )

In [ ]:
history_np = np.asarray(history_q)

plt.figure(figsize=(7,4))
epochs = np.arange(len(history_np[:,0]))
plt.plot(epochs, -history_np[:,3], label="Train rate")
plt.plot(epochs, history_np[:,4], label="Test rate")
plt.axhline(cvx_base, linestyle="--", linewidth=2, label=f"CVX test ({cvx_base:.4f})", color="red")
plt.axhline(cvx_base_train, linestyle=":", linewidth=2, label=f"CVX train ({cvx_base_train:.4f})", color="red")
plt.axhline(wmmse_base, linestyle="-.", linewidth=2, label=f"WMMSE test ({wmmse_base:.4f})", color="blue")
plt.axhline(wmmse_base_train, linestyle=":", linewidth=2, label=f"WMMSE train ({wmmse_base_train:.4f})", color="blue")
plt.xlabel("Epoch")
plt.ylabel(f"{CFG['objective']}-rate")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()